In [ ]:
!pip install yfinance arch --quiet

In [ ]:
import numpy as np
import pandas as pd
import yfinance as yf
import matplotlib.pyplot as plt

from arch import arch_model

In [ ]:
ticker = "^GSPC"

data = yf.download(
    ticker,
    start="2010-01-01",
    end="2024-01-01"
)

data = data[['Close']].dropna()

data.columns = ['Price']

data.head()

In [ ]:
data['Returns'] = 100 * np.log(data['Price'] / data['Price'].shift(1))
data = data.dropna()

returns = data['Returns']

In [ ]:
plt.figure(figsize=(12,5))
plt.plot(returns)
plt.title("S&P 500 Log Returns")
plt.show()

In [ ]:
rolling_vol = returns.rolling(20).std()

plt.figure(figsize=(12,5))
plt.plot(rolling_vol)
plt.title("Rolling 20-Day Volatility")
plt.show()

In [ ]:
def ewma_volatility(returns, lam=0.94):
    vol = np.zeros(len(returns))
    vol[0] = np.var(returns)

    for i in range(1, len(returns)):
        vol[i] = lam * vol[i-1] + (1 - lam) * returns.iloc[i-1]**2

    return np.sqrt(vol)

ewma_vol = ewma_volatility(returns)

In [ ]:
garch_model = arch_model(
    returns,
    vol='GARCH',
    p=1,
    q=1,
    dist='t'
)

garch_result = garch_model.fit(disp='off')

print(garch_result.summary())

In [ ]:
garch_vol = garch_result.conditional_volatility

In [ ]:
plt.figure(figsize=(12,5))

plt.plot(data.index, ewma_vol, label='EWMA Volatility')
plt.plot(data.index, garch_vol, label='GARCH Volatility')

plt.legend()
plt.title("EWMA vs GARCH Volatility")
plt.show()

In [ ]:
z_95 = 1.65
z_99 = 2.33

ewma_var_95 = ewma_vol * z_95
garch_var_95 = garch_vol * z_95

ewma_var_99 = ewma_vol * z_99
garch_var_99 = garch_vol * z_99

In [ ]:
ewma_violations_95 = (returns < -ewma_var_95).sum()
garch_violations_95 = (returns < -garch_var_95).sum()

ewma_violations_99 = (returns < -ewma_var_99).sum()
garch_violations_99 = (returns < -garch_var_99).sum()

print("95% VaR Violations")
print("EWMA:", ewma_violations_95)
print("GARCH:", garch_violations_95)

print("\n99% VaR Violations")
print("EWMA:", ewma_violations_99)
print("GARCH:", garch_violations_99)

In [ ]:
plt.figure(figsize=(12,6))

plt.plot(returns.index, returns, label='Returns', alpha=0.6)
plt.plot(returns.index, -ewma_var_95, label='EWMA VaR 95%')
plt.plot(returns.index, -garch_var_95, label='GARCH VaR 95%')

plt.title('Returns vs Value at Risk (95%)')
plt.xlabel('Date')
plt.ylabel('Returns')

plt.legend()
plt.grid(True)

plt.show()

In [ ]:
params = garch_result.params

alpha = params['alpha[1]']
beta = params['beta[1]']

print("Alpha:", alpha)
print("Beta:", beta)
print("Persistence (alpha + beta):", alpha + beta)